In [1]:
from ppg_basis import ppgGenerator
from ppg_basis import ppgExtractor
import time
import cProfile

In [ ]:
def ppg_generation_gamma():
    start = time.perf_counter()

    # Generate signal
    ppgGen = ppgGenerator(fs=125,
                        hr=60,
                        mu=1,
                        sigma=0,
                        duration=10,
                        L=2,
                        basis_type="gamma")

    sig = ppgGen.generate_signal()
    """plt.plot(sig)
    plt.xlabel("Time")
    plt.ylabel("Intensity")
    plt.title("Original PPG")"""

    end = time.perf_counter()
    print(f"Generate signal runtime: {end - start:.4f} seconds")

    return sig

profiler = cProfile.Profile()
profiler.enable()
sig = ppg_generation_gamma()
profiler.disable()
profiler.dump_stats("gamma/ppg_generation_gamma.prof")

Generate signal runtime: 11.9449 seconds


In [ ]:
def ppg_extraction_gamma():
    start = time.perf_counter()

    # Extract signal parameters
    ppgExt = ppgExtractor(signal=sig,
                        fs=125,
                        hr=60,
                        sigma=0,
                        L=2,
                        basis_type='gamma')

    # ppgExt.plot_cost_landscape()

    ppgExt.mse_flag = True
    ppgExt.corr_flag = True
    ppgExt.appg_flag = False
    theta_pred, params_pred = ppgExt.extract_ppg(block_update=True, 
                                                coord_cycles=4)

    end = time.perf_counter()

    print(f"Extract signal parameters runtime: {end - start:.4f} seconds")

    return theta_pred, params_pred

profiler = cProfile.Profile()
profiler.enable()
theta_pred, params_pred = ppg_extraction_gamma()
profiler.disable()
profiler.dump_stats("gamma/ppg_extraction_gamma.prof")

Extract signal parameters runtime: 273.5163 seconds


In [ ]:
def ppg_extracted_generation_gamma():
    start = time.perf_counter()

    # Generate PPG using extracted parameters
    ppgPrd = ppgGenerator(fs=125,
                        hr=60,
                        mu=1,
                        sigma=0,
                        duration=10,
                        L=2,
                        basis_type="gamma",
                        thetas=theta_pred,
                        params=params_pred)
    pred = ppgPrd.generate_signal()

    """plt.plot(sig)
    plt.plot(pred)
    plt.xlabel("Time")
    plt.ylabel("Intensity")
    plt.title("Original vs Predicted PPG")
    plt.legend(["Original", "Predicted"])"""

    end = time.perf_counter()

    print(f"Generate PPG using extracted parameters runtime: {end - start:.4f} seconds")

    return pred

profiler = cProfile.Profile()
profiler.enable()
pred = ppg_extracted_generation_gamma()
profiler.disable()
profiler.dump_stats("gamma/ppg_extracted_generation_gamma.prof")

Generate PPG using extracted parameters runtime: 0.0398 seconds
